In [1]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
from peft import LoraConfig, get_peft_model

# Load pretrained model (FROZEN BASE)

model_name = "gpt2"

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float32
)

# IMPORTANT: base model is frozen
for param in model.parameters():
    param.requires_grad = False



c:\Users\Arun\Desktop\nitfsds_0825\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
`torch_dtype` is deprecated! Use `dtype` instead!


In [4]:
import warnings
warnings.filterwarnings("ignore")


lora_config = LoraConfig(
    r=8,                      # Rank (low-rank dimension)
    lora_alpha=32,            # Scaling
    target_modules=["c_attn"],# Attention projection layer in GPT-2
    lora_dropout=0.1,
    bias="none",
    task_type="CAUSAL_LM"
)


model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


trainable params: 294,912 || all params: 124,734,720 || trainable%: 0.2364


In [5]:
text = "Deep learning is changing the world."

inputs = tokenizer(
    text,
    return_tensors="pt",
    padding=True
)

outputs = model(**inputs, labels=inputs["input_ids"])
loss = outputs.loss

loss.backward()
print("Loss:", loss.item())



`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Loss: 3.5190107822418213


In [6]:
for name, param in model.named_parameters():
    if param.requires_grad:
        print(name)


base_model.model.base_model.model.transformer.h.0.attn.c_attn.lora_A.default.weight
base_model.model.base_model.model.transformer.h.0.attn.c_attn.lora_B.default.weight
base_model.model.base_model.model.transformer.h.1.attn.c_attn.lora_A.default.weight
base_model.model.base_model.model.transformer.h.1.attn.c_attn.lora_B.default.weight
base_model.model.base_model.model.transformer.h.2.attn.c_attn.lora_A.default.weight
base_model.model.base_model.model.transformer.h.2.attn.c_attn.lora_B.default.weight
base_model.model.base_model.model.transformer.h.3.attn.c_attn.lora_A.default.weight
base_model.model.base_model.model.transformer.h.3.attn.c_attn.lora_B.default.weight
base_model.model.base_model.model.transformer.h.4.attn.c_attn.lora_A.default.weight
base_model.model.base_model.model.transformer.h.4.attn.c_attn.lora_B.default.weight
base_model.model.base_model.model.transformer.h.5.attn.c_attn.lora_A.default.weight
base_model.model.base_model.model.transformer.h.5.attn.c_attn.lora_B.default

In [ ]:
model.save_pretrained("./lora-adapter")
